In [1]:
import os

In [10]:
%pwd

'd:\\projects\\wine-quality-mlops'

In [3]:
os.chdir("..")
%pwd

'd:\\projects\\wine-quality-mlops'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [5]:
from wine_quality_mlops.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from wine_quality_mlops.utils.io_utils import read_yaml, create_directories

In [15]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        self.artifacts_root = Path(self.config["artifacts_root"])

        create_directories([self.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:

        config = self.config["model_trainer"]
        params = self.params["ElasticNet"]
        target_column = self.schema["TARGET_COLUMN"]["name"]

        root_dir = self.artifacts_root / config["root_dir"]

        create_directories([root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=root_dir,
            train_data_path=Path(config["train_data_path"]),
            test_data_path=Path(config["test_data_path"]),
            model_name=config["model_name"],
            alpha=params["alpha"],
            l1_ratio=params["l1_ratio"],
            target_column=target_column
        )

        return model_trainer_config

In [8]:
import os
import sys
import joblib
import pandas as pd
from wine_quality_mlops.utils.logger import logger
from wine_quality_mlops.utils.exceptions import CustomException
from sklearn.linear_model import ElasticNet

In [11]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)


        train_x = train_data.drop([self.config.target_column], axis=1)
        test_x = test_data.drop([self.config.target_column], axis=1)
        train_y = train_data[[self.config.target_column]]
        test_y = test_data[[self.config.target_column]]


        lr = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
        lr.fit(train_x, train_y)

        model_path = (
            self.config.root_dir / self.config.model_name
        )

        joblib.dump(lr, model_path)

In [ ]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise CustomException(e, sys)

[2026-05-16 17:24:36,732] io_utils.py:22 datascienceLogger - INFO - YAML file loaded: D:\projects\wine-quality-mlops\config\config.yaml
[2026-05-16 17:24:36,736] io_utils.py:22 datascienceLogger - INFO - YAML file loaded: D:\projects\wine-quality-mlops\params.yaml
[2026-05-16 17:24:36,738] io_utils.py:22 datascienceLogger - INFO - YAML file loaded: D:\projects\wine-quality-mlops\schema.yaml
[2026-05-16 17:24:36,739] io_utils.py:40 datascienceLogger - INFO - Created directory: artifacts
[2026-05-16 17:24:36,740] io_utils.py:40 datascienceLogger - INFO - Created directory: artifacts\model_trainer


: 